In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import os
import time

In [2]:
# Constants
RandomSTATE = 1082
TRAIN_SIZE = 0.2
MODEL_NAME = "StackingClassifier(LogisticRegression RandomForest SVC)"
DATASET_NAME = "spoofing-merged-gps-only.csv"
RESULTS_CSV = "../../../UAVATTACK_Results.csv"

In [3]:
# Load the dataset
try:
    df = pd.read_csv("../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv")
except FileNotFoundError:
    print("Error: The file '../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv' was not found.")
    exit()

In [4]:
# Data Preprocessing

# Clean up column names
df.columns = df.columns.str.replace('[\\[\\]<>]', '_', regex=True)

# Drop label and timestamp
drop_cols = ['timestamp', 'label']
X = df.drop(columns=drop_cols)
y = df['label']

if y.dtype == object or y.dtype.name == 'category':
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=TRAIN_SIZE, random_state=RandomSTATE, stratify=y
)

In [5]:
# Define base models
base_learners = [
    ('lr', LogisticRegression(random_state=RandomSTATE, max_iter=1200, solver='saga')),
    ('rf', RandomForestClassifier(random_state=RandomSTATE)),
    ('svc', SVC(probability=True, random_state=RandomSTATE))
]

# Meta model
meta_learner = LogisticRegression(random_state=RandomSTATE, max_iter=1200, solver='saga')

In [6]:
# Create and train the Stacking model
stacking_model = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    passthrough=False,
    n_jobs=-1
)
start_time = time.time()
stacking_model.fit(X_train, y_train)
train_time = time.time() - start_time

In [7]:
# Predict and evaluate
y_pred = stacking_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
print(f"Stacked Model Accuracy on Test Set: {accuracy * 100:.2f}%")
print(f"F1 Score: {f1:.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Stacked Model Accuracy on Test Set: 99.83%
F1 Score: 0.9983
Confusion Matrix:
[[2500    0]
 [   5  393]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2500
           1       1.00      0.99      0.99       398

    accuracy                           1.00      2898
   macro avg       1.00      0.99      1.00      2898
weighted avg       1.00      1.00      1.00      2898



In [8]:
# Save results to CSV
results = [{
    "model": MODEL_NAME,
    "dataset": DATASET_NAME,
    "f1score": f1,
    "accuracy": accuracy,
    "train_time_sec": train_time,
    "parameters": str(stacking_model.get_params()),
    "num_epochs": "N/A"
}]
results_df = pd.DataFrame(results)

file_exists = os.path.isfile(RESULTS_CSV)
results_df.to_csv(
    RESULTS_CSV,
    mode='a',
    header=not file_exists,
    index=False
)
print(f"Results appended to {RESULTS_CSV}")

Results appended to ../../../UAVATTACK_Results.csv
